## Configuration and test of floweaver Sankey(s) for IEDC dataset 1_F_Global_Material_Flows_ewMFA_Schandl_2020

In [1]:
import pandas as pd
flows = pd.read_excel('1_F_Global_Material_Flows_ewMFA_Schandl_2020.xlsx','Data') # Insert filename of IEDC download
flows

,id,dataset_name,aspect 1 : commodity,aspect 2 : material,aspect 3 : layer,aspect 4 : region,aspect 5 : time,aspect 6 : origin_process,aspect 7 : destination_process,aspect 8 : scenario,...,unitcode,unitcode1,stats_array_1,stats_array_2,stats_array_3,stats_array_4,comment,reserve1,reserve2,reserve3
0,3864022,1_F_Global_Material_Flows_ewMFA_Schandl_2020,balancing item,all materials,dry mass,Global,2020,balancing items input,use as energy carrier,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3864023,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,domestic extraction,material processing,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3864024,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,waste management,material processing,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3864025,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,material processing,"markets, global",History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3864026,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,"markets, global",domestic material consumption,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3864027,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,material processing,domestic material consumption,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,3864028,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,domestic material consumption,use as energy carrier,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,3864029,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,domestic material consumption,use as material,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,3864030,1_F_Global_Material_Flows_ewMFA_Schandl_2020,balancing item,all materials,dry mass,Global,2020,use as energy carrier,balancing items output,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,3864031,1_F_Global_Material_Flows_ewMFA_Schandl_2020,all commodities,all materials,dry mass,Global,2020,use as energy carrier,waste management,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import pandas as pd
flows       = pd.read_excel('1_F_Global_Material_Flows_ewMFA_Schandl_2020.xlsx','Data')
material    = 'all materials'
layer       = 'dry mass'
region      = 'Global'
scenario    = 'History'
time        =  2020

ExtractData = flows.loc[(flows['aspect 2 : material'] == material) & (flows['aspect 3 : layer'] == layer) & (flows['aspect 4 : region'] == region) & (flows['aspect 8 : scenario'] == scenario) & (flows['aspect 5 : time'] == time)]
SankeyDatar = ExtractData.drop(columns=['id','dataset_name','aspect 2 : material','aspect 3 : layer','aspect 4 : region','aspect 8 : scenario','aspect 5 : time','aspect 9 : ','aspect 10 : ','aspect 11 : ','aspect 12 : ','unitcode','unitcode1','stats_array_1','stats_array_2','stats_array_3','stats_array_4','comment','reserve1','reserve2','reserve3'])
SankeyDatar.rename(columns={'aspect 6 : origin_process': 'source', 'aspect 7 : destination_process': 'target','aspect 1 : commodity': 'type'}, inplace=True)
column_names= ['source','target','type','value']
SankeyDatar = SankeyDatar.reindex(columns=column_names)
SankeyDatar.reset_index(drop=True, inplace=True)
SankeyDatar

,source,target,type,value
0,balancing items input,use as energy carrier,balancing item,54.320
1,domestic extraction,material processing,all commodities,94.665
2,waste management,material processing,all commodities,9.343
3,material processing,"markets, global",all commodities,14.087
4,"markets, global",domestic material consumption,all commodities,14.087
5,material processing,domestic material consumption,all commodities,89.921
6,domestic material consumption,use as energy carrier,all commodities,39.751
7,domestic material consumption,use as material,all commodities,64.257
8,use as energy carrier,balancing items output,balancing item,46.989
9,use as energy carrier,waste management,all commodities,47.083


In [3]:
from floweaver import *

# Define the Sankey version, which will be the name of the .json spec file. 
# Default: Sankey version = IEDC template filename without 1_F_ and without .xlxs, start with capital letter
Sankey_version = 'Global_Material_Flows_ewMFA_Schandl_2020'

# Define the nodes
nodes = {
    'balancing items input': ProcessGroup(['balancing items input'], title='Balancing items input'),
    'domestic extraction': ProcessGroup(['domestic extraction'], title='Material extraction'),
    'material processing': ProcessGroup(['material processing'], title='Material input'),  
    'domestic material consumption': ProcessGroup(['domestic material consumption'], title='Material consumption'),
    'markets, global': ProcessGroup(['markets, global'], title='International trade'), 
    'use as energy carrier': ProcessGroup(['use as energy carrier'], title='Energetic use'), 
    'use as material': ProcessGroup(['use as material'], title='Technical use'), 
    'use phase': ProcessGroup(['use phase'], title='Use phase'), 
    'balancing items output': ProcessGroup(['balancing items output'], title='Balancing items output'), 
    'waste management': ProcessGroup(['waste management'], title='Domestic processed output'),
    'atmosphere': ProcessGroup(['atmosphere'], title='Air emissions'), 
    'dissipative use, losses': ProcessGroup(['dissipative use, losses'], title='Dissipative uses of products'), 
    'treatment or disposal of solid and liquid waste': ProcessGroup(['treatment or disposal of solid and liquid waste'], title='Solid and liquid waste'),
    'loop1': Waypoint(title='Recycling', direction='L'),
    'loop2': Waypoint(title=' ', direction='L'),
}
# Define the bundles
bundles = [
    Bundle('domestic extraction', 'material processing'),
    Bundle('material processing', 'markets, global'),
    Bundle('markets, global', 'domestic material consumption'),
    Bundle('material processing', 'domestic material consumption'),
    Bundle('domestic material consumption', 'use as energy carrier'),
    Bundle('domestic material consumption', 'use as material'),
    Bundle('balancing items input', 'use as energy carrier'),
    Bundle('use as energy carrier', 'balancing items output'),
    Bundle('use as energy carrier','waste management'),
    Bundle('use as material', 'use phase'), 
    Bundle('use as material', 'waste management'),
    Bundle('use phase', 'waste management'),
    Bundle('waste management', 'material processing', waypoints=['loop2','loop1']),
    Bundle('waste management', 'atmosphere'),
    Bundle('waste management', 'treatment or disposal of solid and liquid waste'),
    Bundle('waste management', 'dissipative use, losses'),
]

# Define the ordering
ordering = [
[['balancing items input'],[],['domestic extraction'],[],[]],
[[],[],['material processing'],[],['loop1']],
[[],['markets, global'],[],[],[]],
[[],[],['domestic material consumption'],[],[]],
[['use as energy carrier'],[],['use as material'],[],[]],  
[[],[],['use phase'],[],[]],
[[],[],['waste management'],[],['loop2']], 
[['balancing items output'],['atmosphere'],['treatment or disposal of solid and liquid waste'],['dissipative use, losses'],[]],     
]

flows_by_type = Partition.Simple('type', ['all commodities','balancing item','solid and liquid waste','emissions to air'])

palette = {
    'all commodities':                         'midnightblue',
    'balancing item':                          'skyblue',
    'solid and liquid waste':                  'lightsteelblue',
    'emissions to air':                        'steelblue',
}

# Specify canvas size and scale (thickness) of the flows:
size = dict(width=1000, height=200)
SCALE = 0.3

# New SDD with the flow_partition set
sdd = SankeyDefinition(nodes, bundles, ordering, flow_partition=flows_by_type)

Sankey = weave(sdd, SankeyDatar, palette=palette).to_widget(**size)
Sankey.scale = SCALE

Sankey # Show below

Sankey.auto_save_png(Sankey_version + '_' + region + '_' + str(time) + '.png')
Sankey.auto_save_svg(Sankey_version + '_' + region + '_' + str(time) + '.svg')

In [4]:
# double-check and export Sankey specification as .json
if isinstance(SankeyDatar, pd.DataFrame):
    SankeyDatar = Dataset(SankeyDatar)
    
spec = compile_sankey_definition(
    sdd,       # see above: sdd = SankeyDefinition(nodes, bundles, ordering, flow_partition=flows_by_type)
    measures   = "value",
    link_width = None,
    link_color = None,
    palette    = palette,
    add_elsewhere_waypoints = True,
    dimension_tables        = None,
)

Sankey = execute_weave(spec, SankeyDatar)
Sankey.to_widget(**size)

In [5]:
# export Sankey spec as .json for use in IEDC web application
import json
json_data = json.dumps(spec.to_json(), indent=2)
# Write Sankey spec to .json, filename as indicated in: 
# https://github.com/IndEcol/IEDC_floweaver_integration/blob/main/floweaver_IEDC_Sankey_configs/floweaver_IEDC_Sankey_configs.config
with open(Sankey_version + '.json', "w", encoding="utf-8") as f:
    f.write(json_data)
    f.write("\n")